# **📄 Clean & Jerk Dataset — Publication Figures & Tables (`Plan.md`)**

### 🔧 Colab Setup & Library Init

In [ ]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

if importlib.util.find_spec("h5py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py"])

repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "dataset_explorer" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "checkout", git_branch], check=False)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
        for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
            importlib.reload(sys.modules[_m])
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import display_table
from src.explorer import (
    DEFAULT_CLEAN_DATASET_DIR,
    build_paper_tables,
    plot_fig4_annotation_scheme,
)

dataset_dir = DEFAULT_CLEAN_DATASET_DIR # @param {type:"string"}
EXPORT_CFG = {"toImageButtonOptions": {"format": "png", "scale": 4}}

### 📊 Paper Tables (Table 5 Demographics, Table 7 Coverage, Table 8 Agreement)

In [ ]:
# @title 📊 Generate Paper Tables 5, 7, 8
paper_tables = build_paper_tables(dataset_dir)
for name, df in paper_tables.items():
    print(f"\n=== {name} ===")
    display_table(df, clean_columns=False)

### 🖼️ Figure 4 — 11-Phase Annotation Scheme

In [ ]:
# @title 🖼️ Figure 4: Annotation Scheme { display-mode: "form" }
sample_set_id = "set_0003" # @param {type:"string"}
sample_lift_id = 1 # @param {type:"integer"}
sample_sensor = "US" # @param ["RKP","RHD","REP","RSD","LKP","LHD","LEP","LSD","LHP","RHP","LS","RWD","RWP","RED","LWD","LWP","LED","LAD","LAP","LKD","LSP","RSP","US","RAD","RAP","RKD"]
modality = "gyro" # @param ["gyro", "accel", "both"]
signal_mode = "axes" # @param ["axes", "magnitude"]
axes_str = "X,Y,Z" # @param {type:"string"}  # e.g. "X,Y,Z" or "Y" or "X,Z"

selected_axes = [a.strip() for a in axes_str.split(",") if a.strip()]
fig4 = plot_fig4_annotation_scheme(
    dataset_dir,
    set_id=sample_set_id,
    lift_id=sample_lift_id,
    sensor_code=sample_sensor,
    modality=modality,
    signal_mode=signal_mode,
    axes=selected_axes,
)
fig4.show(config=EXPORT_CFG)